# Online Shopper Purchase Intention Prediction

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `purchased`

## 2 · Project Overview

We predict whether an online shopping session will result in a **purchase** based on browsing behavior, visitor type, timing, and engagement metrics.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a shopping session's page views, duration, bounce/exit rates, product page visits, visitor type, month, and weekend flag, predict purchase intention.

## 5 · Why This Project Matters

- **E-commerce conversion** optimization is directly tied to revenue.
- Understanding purchase intent enables real-time personalization.
- Reducing cart abandonment is a key e-commerce challenge.
- Purchase prediction drives ad targeting and retargeting strategies.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 9,000 |
| **Features** | page_views, duration_min, bounce_rate, exit_rate, product_related_pages, month, visitor_type, weekend |
| **Target** | purchased (0 = no purchase, 1 = purchased) |
| **Class balance** | ~75/25 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "purchased"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: purchased
Save dir: E:\Github\Machine-Learning-Projects\Classification\Online Shopper Purchase Intention Prediction


## 11 · Dataset Download or Loading

Synthetic dataset: 9,000 online shopping sessions with browsing behavior and purchase outcome.

In [4]:
np.random.seed(SEED)
n = 9000
page_views = np.random.poisson(5, n)
duration_min = np.round(np.random.exponential(10, n).clip(0.5, 120), 1)
bounce_rate = np.round(np.random.beta(2, 5, n), 3)
exit_rate = np.round(np.random.beta(2, 4, n), 3)
product_related_pages = np.random.poisson(3, n)
month = np.random.choice(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                           "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], n)
visitor_type = np.random.choice(["Returning", "New", "Other"], n, p=[0.6, 0.35, 0.05])
weekend = np.random.binomial(1, 0.25, n)
vis_num = np.where(visitor_type == "Returning", 2, np.where(visitor_type == "New", 1, 0))

score = (0.03 * page_views + 0.01 * duration_min - 1.5 * bounce_rate
         - 1.0 * exit_rate + 0.08 * product_related_pages + 0.15 * vis_num
         - 0.1 * weekend + np.random.normal(0, 0.7, n))
purchased = (score > np.percentile(score, 75)).astype(int)

df = pd.DataFrame({"page_views": page_views, "duration_min": duration_min,
                    "bounce_rate": bounce_rate, "exit_rate": exit_rate,
                    "product_related_pages": product_related_pages,
                    "month": month, "visitor_type": visitor_type,
                    "weekend": weekend, "purchased": purchased})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['purchased'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (9000, 9)
Class balance:
purchased
0    0.75
1    0.25
Name: proportion, dtype: float64


,page_views,duration_min,bounce_rate,exit_rate,product_related_pages,month,visitor_type,weekend,purchased
0,5,16.4,0.101,0.302,8,Oct,Returning,1,0
1,4,6.6,0.698,0.483,3,Oct,New,0,0
2,4,33.7,0.123,0.185,2,Jun,Returning,0,0
3,5,1.2,0.231,0.556,8,Jul,New,1,0
4,5,1.4,0.250,0.037,1,Sep,Returning,0,0


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (9000, 9)

Missing values:
page_views               0
duration_min             0
bounce_rate              0
exit_rate                0
product_related_pages    0
month                    0
visitor_type             0
weekend                  0
purchased                0
dtype: int64

Duplicate rows: 0

Dtypes:
page_views                 int32
duration_min             float64
bounce_rate              float64
exit_rate                float64
product_related_pages      int32
month                     object
visitor_type              object
weekend                    int32
purchased                  int64
dtype: object

Target 'purchased' confirmed. Value counts:
purchased
0    6750
1    2250
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["page_views", "duration_min", "bounce_rate", "exit_rate", "product_related_pages"]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
axes[1][2].axis("off")
plt.suptitle("Feature Distributions by Purchase Status", fontsize=14)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df.groupby("visitor_type")[TARGET].mean().plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Purchase Rate by Visitor Type")
df.groupby("weekend")[TARGET].mean().plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="black")
axes[1].set_title("Purchase Rate by Weekend")
axes[1].set_xticklabels(["Weekday", "Weekend"], rotation=0)
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `purchased`.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind="bar", ax=axes[0], color=["salmon", "steelblue"], edgecolor="black")
axes[0].set_title("Purchase Distribution")
axes[0].set_xticklabels(["No Purchase (0)", "Purchased (1)"], rotation=0)

monthly_rate = df.groupby("month")[TARGET].mean()
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
monthly_rate = monthly_rate.reindex(month_order)
axes[1].plot(monthly_rate.index, monthly_rate.values, marker="o", color="steelblue")
axes[1].set_title("Purchase Rate by Month")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()
print(f"Overall purchase rate: {df[TARGET].mean():.1%}")

Overall purchase rate: 25.0%


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (7200, 8), Test: (1800, 8)
Train class distribution:
purchased
0    0.75
1    0.25
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `month`, `visitor_type` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Class imbalance**: ~25% purchased — moderate imbalance, stratified split used.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["engagement_score"] = X_train["page_views"] * X_train["duration_min"] / (1 + X_train["bounce_rate"])
X_test["engagement_score"] = X_test["page_views"] * X_test["duration_min"] / (1 + X_test["bounce_rate"])

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (9): ['page_views', 'duration_min', 'bounce_rate', 'exit_rate', 'product_related_pages', 'month', 'visitor_type', 'weekend', 'engagement_score']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.7611
F1       : 0.2456

              precision    recall  f1-score   support

           0       0.77      0.96      0.86      1350
           1       0.58      0.16      0.25       450

    accuracy                           0.76      1800
   macro avg       0.68      0.56      0.55      1800
weighted avg       0.73      0.76      0.70      1800



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                             Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                        
NearestCentroid              0.682778           0.670000  0.727086  0.701110   0.744234  0.682778    0.019245
PassiveAggressiveClassifier  0.648889           0.594074  0.623080  0.664856   0.690980  0.648889    0.021693
XGBClassifier                0.739444           0.567778  0.658556  0.705169   0.698484  0.739444    0.169771
Perceptron                   0.647778           0.567407  0.596780  0.658615   0.672982  0.647778    0.018638
DecisionTreeClassifier       0.670556           0.566296  0.566296  0.672336   0.674200  0.670556    0.051514
ExtraTreeClassifier          0.666667           0.564444  0.564444  0.669501   0.672547  0.666667    0.021321
LabelSpreading               0.681667           0.562593  0.606577  0.677263   0.67333

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='binary'):.4f}")

FLAML best model: lgbm
Accuracy: 0.7017
F1: 0.3107


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.2242  (1.3s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[80]	valid_0's binary_logloss: 0.511565
LightGBM F1: 0.2351  (1.0s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="binary"), 4),
        "Precision": round(precision_score(y_test, yp, average="binary", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="binary", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
FLAML                  0.7017  0.3107     0.3678  0.2689
Logistic Regression    0.7611  0.2456     0.5833  0.1556
LightGBM               0.7506  0.2351     0.5036  0.1533
CatBoost               0.7539  0.2242     0.5289  0.1422

Top 3 models: ['FLAML', 'Logistic Regression', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='binary'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='binary', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='binary', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  FLAML:
    Accuracy : 0.7017
    F1       : 0.3107
    Precision: 0.3678
    Recall   : 0.2689

  Logistic Regression:
    Accuracy : 0.7611
    F1       : 0.2456
    Precision: 0.5833
    Recall   : 0.1556

  LightGBM:
    Accuracy : 0.7506
    F1       : 0.2351
    Precision: 0.5036
    Recall   : 0.1533


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: FLAML

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.85      0.81      1350
           1       0.37      0.27      0.31       450

    accuracy                           0.70      1800
   macro avg       0.57      0.56      0.56      1800
weighted avg       0.67      0.70      0.68      1800


Total errors: 537 / 1800 (29.8%)

Sample misclassifications:
      page_views  duration_min  bounce_rate  exit_rate  product_related_pages  month  visitor_type  weekend  engagement_score  actual  predicted  correct
4845           6          32.0        0.205      0.329                      6    7.0           2.0        0        159.336100       1          0    False
2237           9           1.0        0.329      0.371                      0   11.0           2.0        0          6.772009       1          0    False
7698          10           5.1        0.021      0.071                      0   11.0           0.0        0

## 25 · Interpretation and Business Insight

**Key findings:**
- **Bounce rate** and **exit rate** are the strongest negative predictors.
- **Product-related page views** strongly indicate purchase intent.
- **Returning visitors** convert at higher rates.
- **Weekend** sessions have slightly lower conversion.

**Business takeaway:** Reduce bounce rates, surface relevant products, and prioritize returning visitor engagement.

## 26 · Limitations

1. Synthetic data with simplified browsing patterns.
2. No product category or pricing information.
3. Missing cart-level data.
4. No user device or browser details.
5. Real purchase rates vary widely by industry.

## 27 · How to Improve This Project

1. Use real Google Analytics / e-commerce data.
2. Add product category and price features.
3. Include cart abandonment signals.
4. Add referral source and campaign data.
5. Model as a funnel (view → cart → purchase).

## 28 · Production Considerations

- Deploy for real-time session scoring.
- Trigger pop-ups or offers for high-intent sessions.
- Integrate with recommendation engine.
- Monitor conversion rate drift weekly.
- A/B test interventions for medium-probability sessions.

## 29 · Common Mistakes

1. Ignoring class imbalance (75/25 split).
2. Not considering session duration bias.
3. Using accuracy alone for evaluation.
4. Including post-purchase data (leakage).
5. Not segmenting by visitor type in analysis.

## 30 · Mini Challenge / Exercises

1. Remove `bounce_rate` — how much does F1 change?
2. Create `pages_per_minute = page_views / duration_min` and test.
3. Segment by visitor_type and compare model performance.
4. Try SMOTE and compare with class_weight approach.
5. Plot monthly conversion trends and discuss seasonality.

## 31 · Final Summary / Key Takeaways

1. **Bounce and exit rates** are the strongest purchase signals (negative).
2. **Product page engagement** indicates purchase intent.
3. **Returning visitors** convert better than new ones.
4. **Class imbalance** requires careful metric selection.
5. **Real-time scoring** enables personalized interventions.
6. **Engagement features** (composite scores) add predictive value.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="binary")), 4),
        "precision": round(float(precision_score(y_test, yp, average="binary", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="binary", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Online Shopper Purchase Intention Prediction\metrics.json
{
  "CatBoost": {
    "accuracy": 0.7539,
    "f1": 0.2242,
    "precision": 0.5289,
    "recall": 0.1422
  },
  "LightGBM": {
    "accuracy": 0.7506,
    "f1": 0.2351,
    "precision": 0.5036,
    "recall": 0.1533
  },
  "Logistic Regression": {
    "accuracy": 0.7611,
    "f1": 0.2456,
    "precision": 0.5833,
    "recall": 0.1556
  },
  "FLAML": {
    "accuracy": 0.7017,
    "f1": 0.3107,
    "precision": 0.3678,
    "recall": 0.2689
  }
}
